In [21]:
import json
import pickle

def read_json(file_path):    
    with open(file_path) as f:
        json_data = json.load(f)

    return json_data

def read_pickle(file_path):
    with open(file_path, 'rb') as f:
        pickle_data = pickle.load(f)

    return pickle_data

def read_jsonl(filepath):
    data = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data

def save_json(data, filepath, indent=2):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)


def save_pickle(data, filepath, protocol=pickle.HIGHEST_PROTOCOL):
    with open(filepath, "wb") as f:
        pickle.dump(data, f, protocol=protocol)

def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False))
            f.write("\n")

In [4]:
data1 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160290.json")
data2 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160291.json")
data3 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160292.json")

In [6]:
print(len(data1[0]))

1000


In [12]:
core_data = dict()
core_data['shots'] = dict()
core_data['queries'] = dict()
core_data['umbc_shots'] = dict()

def extract_shot(shot):
    new_shot = dict()

    new_shot["id"] = shot["id"]
    new_shot['relation'] = shot['relation']
    new_shot['token'] = shot['token'].copy()
    new_shot['subj_start'] = shot['subj_start']
    new_shot['subj_end'] = shot['subj_end']
    new_shot['obj_start'] = shot['obj_start']
    new_shot['obj_end'] = shot['obj_end']
    new_shot['subj_type'] = shot['subj_type']
    new_shot['obj_type'] = shot['obj_type']
    
    return new_shot

episode_files = []
new_episodes = []

for cur_data in [data1, data2, data3]:
    cur_data = cur_data[0]
    # print(len(cur_data))

    
    for q_ in range(3):
        for episode in cur_data:
            new_episode = dict()

            train_ = episode['meta_train']
            test_ = episode['meta_test'][q_]

            new_ways = []
            for way in train_:
                new_shots = []
                for uidx, shot in enumerate(way):
                    
                    assert shot['token'] == shot['tokens']

                    if uidx > 0:
                        id_ = shot['id']
                        if id_ not in core_data['umbc_shots']:
                            new_shot = extract_shot(shot)
                            core_data['umbc_shots'][id_] = new_shot

                    else:
                        id_ = shot['id']
                        if id_ not in core_data['shots']:
                            new_shot = extract_shot(shot)
                            core_data['shots'][id_] = new_shot
        
                    new_shots.append(id_)
                new_ways.append(new_shots)


            id_ = test_['id']
            if id_ not in core_data['queries']:
                new_shot = extract_shot(test_)
                core_data['queries'][id_] = new_shot

            new_q = [id_]
            new_episode['meta_train'] = new_ways
            new_episode['meta_test'] = new_q

            new_episodes.append(new_episode)

    print(len(new_episodes))

print(len(core_data['shots']))
print(len(core_data['queries']))
print(len(core_data['umbc_shots']))
    

3000
6000
9000
633
7365
0


In [13]:
save_pickle(core_data, "../data/fs_tacred_dev_shots_details.pkl")

In [14]:
save_pickle(new_episodes, "../data/fs_tacred_dev_episodes_1shots.pkl")

In [22]:
test_data = read_pickle("../data/fs_tacred_test_episodes_shots_details.pkl")

In [23]:
test_data_copy = dict()

test_data_copy["shots"] = test_data["shots"]
test_data_copy["queries"] = test_data["queries"]
test_data_copy["umbc_shots"] = dict()

In [24]:
save_pickle(test_data_copy, "../data/fs_tacred_test_episodes_shots_details.pkl")